In [1]:
import os
import time
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
import keras_cv
import tensorflow as tf
import numpy as np
import pandas as pd
import librosa
import random
from glob import glob
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# === CONFIGURATION ===
class CFG:
    seed = 42
    img_size = [128, 384]
    duration = 10
    sample_rate = 32000
    audio_len = duration * sample_rate
    nfft = 2028
    hop_length = audio_len // (img_size[1] - 1)
    fmin = 20
    fmax = 16000
    preset = 'efficientnetv2_b2_imagenet'
    DEBUG_MODE = False
    DEBUG_PCT = 0.01 # not used right now

    BASE_PATH = '/kaggle/input/birdclef-2025'
    MODEL_PATH = "/kaggle/input/team-model-bird/best_model.weights.h5"

    class_names = sorted(os.listdir(f'{BASE_PATH}/train_audio'))
    num_classes = len(class_names)
    name2label = {name: idx for idx, name in enumerate(class_names)}
    label2name = {idx: name for name, idx in name2label.items()}

random.seed(CFG.seed)
tf.keras.utils.set_random_seed(CFG.seed)

# === MODEL SETUP ===
start = time.time()
inp = keras.layers.Input(shape=(CFG.img_size[0], CFG.img_size[1], 3))
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
out = keras_cv.models.ImageClassifier(backbone=backbone, num_classes=CFG.num_classes)(inp)
model = keras.models.Model(inputs=inp, outputs=out)
model.load_weights(CFG.MODEL_PATH)
model.compile()
print("✅ Model loaded in {:.2f} seconds".format(time.time() - start))

# === APPLY POST-STANDARDIZATION TO MEL SPEC ===
def apply_preproc(spec):
    mean = tf.math.reduce_mean(spec)
    std = tf.math.reduce_std(spec)
    spec = tf.where(tf.math.equal(std, 0), spec - mean, (spec - mean) / std)

    min_val = tf.math.reduce_min(spec)
    max_val = tf.math.reduce_max(spec)
    spec = tf.where(
        tf.math.equal(max_val - min_val, 0),
        spec - min_val,
        (spec - min_val) / (max_val - min_val),
    )
    return spec

# === MEL-SPECTROGRAM PREPROCESSING ===
def preprocess_window(audio):
    if len(audio) < CFG.audio_len:
        audio = np.pad(audio, (0, CFG.audio_len - len(audio)))
    elif len(audio) > CFG.audio_len:
        audio = audio[:CFG.audio_len]

    spec = librosa.feature.melspectrogram(
        y=audio, sr=CFG.sample_rate, n_fft=CFG.nfft,
        hop_length=CFG.hop_length, n_mels=CFG.img_size[0],
        fmin=CFG.fmin, fmax=CFG.fmax
    )
    db = librosa.power_to_db(spec, ref=1.0)
    db = apply_preproc(db)  # Apply post-processing normalization
    db = np.stack([db] * 3, axis=-1)
    db = tf.image.resize(db, CFG.img_size).numpy()
    return db

# === GATHER TEST FILES ===
def get_test_files():
    # if CFG.DEBUG_MODE:
    #     print("🔍 DEBUG MODE ON: Using small subset of train_soundscapes")
    #     debug_files = sorted(Path(f'{CFG.BASE_PATH}/train_soundscapes').glob("*.ogg"))
    #     random.shuffle(debug_files)
    #     num_debug = max(1, int(CFG.DEBUG_PCT * len(debug_files)))
    #     debug_subset = debug_files[:num_debug]
    #     print(f"🧪 Selected {num_debug} debug files from train_soundscapes")
    #     return debug_subset
    if CFG.DEBUG_MODE:
        print("🔍 DEBUG MODE ON: Using one .ogg file per species from train_audio")
        sound_files = []
        train_audio_path = Path(f"{CFG.BASE_PATH}/train_audio")
        for class_dir in sorted(os.listdir(train_audio_path)):
            class_path = os.path.join(train_audio_path, class_dir)
            all_audio = [f for f in os.listdir(class_path) if f.endswith('.ogg')]
            if all_audio:
                sound_files.append(os.path.join(class_path, all_audio[0]))  # use one per species
            if len(sound_files) >= 10:
                break  # only a few files for speed
        print(f"🧪 Selected {len(sound_files)} debug files from train_audio")
        return sound_files
    else:
        print("📁 Loading from test_soundscapes for actual prediction")
        test_files = sorted(Path(f'{CFG.BASE_PATH}/test_soundscapes').glob("*.ogg"))
        print(f"✅ Found {len(test_files)} test files")
        return test_files

# === EXTRACT CONTEXT-AWARE 10s WINDOWS PER 5s CHUNK ===
def extract_all_windows(audio_path):
    y, _ = librosa.load(audio_path, sr=CFG.sample_rate)
    file_id = Path(audio_path).stem
    specs, ids = [], []

    chunk_len = CFG.sample_rate * 5  # 5-second target prediction chunks
    context = CFG.audio_len // 2     # 5 seconds before/after = 10s total window

    for i in range(0, len(y), chunk_len):
        center = i + chunk_len // 2
        start = max(0, center - context)
        end = start + CFG.audio_len

        # ✅ Pad if audio runs out near the end
        if end > len(y):
            pad_len = end - len(y)
            audio_window = np.pad(y[start:], (0, pad_len), mode='constant')
        else:
            audio_window = y[start:end]

        if len(audio_window) == CFG.audio_len:
            spec = preprocess_window(audio_window)
            specs.append(spec)
            chunk_end = (i // chunk_len + 1) * 5
            ids.append(f"{file_id}_{chunk_end}")

    return specs, ids

# === INFERENCE RUNNER ===
def run_inference():
    print("\n📂 Loading audio files...")
    start = time.time()
    test_files = get_test_files()
    print(f"✅ Loaded {len(test_files)} files in {time.time() - start:.2f} seconds")

    all_specs, all_ids = [], []
    print("\n🎛️ Precomputing Mel Spectrograms in parallel...")
    start = time.time()
    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(tqdm(executor.map(extract_all_windows, test_files), total=len(test_files)))
        for specs, ids in results:
            all_specs.extend(specs)
            all_ids.extend(ids)
    print(f"✅ Preprocessing done in {time.time() - start:.2f} seconds")

    print("\n🧠 Running predictions...")
    start = time.time()
    if len(all_specs) == 0:
        print("⚠️ No spectrograms found. Skipping prediction.")
        return [], np.zeros((0, CFG.num_classes), dtype=np.float32)
    else:
        batched_input = np.stack(all_specs)
        preds = model.predict(batched_input, batch_size=32, verbose=1)
        print(f"✅ Predictions done in {time.time() - start:.2f} seconds")
        return all_ids, preds

# === SAVE PREDICTIONS TO CSV ===
def save_submission(row_ids, preds):
    print("\n💾 Saving submission file...")
    start = time.time()
    columns = ["row_id"] + CFG.class_names
    df = pd.DataFrame([[rid] + list(prob) for rid, prob in zip(row_ids, preds)], columns=columns)
    df.to_csv("submission.csv", index=False)
    print(f"✅ submission.csv saved in {time.time() - start:.2f} seconds")
    print(df.head())

# === RUN ENTRYPOINT ===
if __name__ == '__main__':
    row_ids, preds = run_inference()
    save_submission(row_ids, preds)


2025-04-13 20:50:50.066942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744577450.301326      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744577450.366985      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-13 20:51:14.284245: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


✅ Model loaded in 10.22 seconds

📂 Loading audio files...
📁 Loading from test_soundscapes for actual prediction
✅ Found 0 test files
✅ Loaded 0 files in 0.00 seconds

🎛️ Precomputing Mel Spectrograms in parallel...


0it [00:00, ?it/s]

✅ Preprocessing done in 0.00 seconds

🧠 Running predictions...
⚠️ No spectrograms found. Skipping prediction.

💾 Saving submission file...
✅ submission.csv saved in 0.02 seconds
Empty DataFrame
Columns: [row_id, 1139490, 1192948, 1194042, 126247, 1346504, 134933, 135045, 1462711, 1462737, 1564122, 21038, 21116, 21211, 22333, 22973, 22976, 24272, 24292, 24322, 41663, 41778, 41970, 42007, 42087, 42113, 46010, 47067, 476537, 476538, 48124, 50186, 517119, 523060, 528041, 52884, 548639, 555086, 555142, 566513, 64862, 65336, 65344, 65349, 65373, 65419, 65448, 65547, 65962, 66016, 66531, 66578, 66893, 67082, 67252, 714022, 715170, 787625, 81930, 868458, 963335, amakin1, amekes, ampkin1, anhing, babwar, bafibi1, banana, baymac, bbwduc, bicwre1, bkcdon, bkmtou1, blbgra1, blbwre1, blcant4, blchaw1, blcjay1, blctit1, blhpar1, blkvul, bobfly1, bobher1, brtpar1, bubcur1, bubwre1, bucmot3, bugtan, butsal1, cargra1, cattyr, chbant1, chfmac1, cinbec1, cocher1, cocwoo1, colara1, colcha1, compau, compot